In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
import numpy as np
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    response=requests.get(url)
    response.raise_for_status()
    return response.json()

In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url: str) -> list:
    list_of_orders=[]
    while first_url:
        data=get_json(first_url)
        next_page = data.get('next')
        list_of_orders.extend(data.get('data', []))
        if next_page:
            first_url = urljoin(first_url, next_page)
        else:
            first_url = None

    return list_of_orders



In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records: list, path: str) -> None:
    with open(path, mode="w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


In [0]:
# 4. Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити у
# задатку 5

def download_file(url: str, path: str) -> int:
    #Download a file and save it unchanged. Returns its size in bytes.
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes, unchanged
    return len(response.content)



In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams','matches'
# 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
# 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције
# download_file
# 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
# Након тога преименовати колоне
# customer_name у customer
# customer_city у city
# 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
# По потреби, pd.read_csv(...) или pd.read_parquet(...)
# 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)

def extract(raw_dir: str) -> dict:
    orders=fetch_all(BASE_URL + 'orders/page-1.json') #vraca listu recnika
    save_json(orders, os.path.join(raw_dir, 'orders.json'))
    download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv')) 
    download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))
    with open(os.path.join(raw_dir, 'orders.json'), 'r', encoding='utf-8') as f:
        orders = json.load(f)
        orders=pd.json_normalize(orders, sep='_')
        orders=orders.rename(columns={'customer_name': 'customer', 'customer_city': 'city'})
    products=pd.read_csv(os.path.join(raw_dir,'products.csv'))
    teams=pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    matches=pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))
    data = {
        'orders': orders,
        'products': products,
        'teams': teams,
        'matches': matches
    }
    return data
 

In [0]:
#pomocna funkcija
def remove_duplicate_products(df: pd.DataFrame) -> pd.DataFrame:
    is_dup = df.duplicated('product_id', keep='first')
    return df[~is_dup]


In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame
# која чисти каталог производа
# Ако се понавља исти производ, садржати први ред за сваки product_id
# team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред
# остаје празан (то су производи лиге, без тима)
# category треба да буде без размака, има мала слова
# price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане
# само float (у сваком случају)
# Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
# active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало
# постаје False
# Output: 105 редова, price је типа float64, active је True за 101 производ
# Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves',
# 'price': 1490.0, 'active': True, ...}

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df=df.copy()
    data=remove_duplicate_products(df)
    data['team_code']=data['team_code'].str.strip().str.upper()
    data['category']=data['category'].str.strip().str.lower()
    data['price'].str.replace(' RSD', '', regex=False)
    data['price']=data['price'].str.replace(',', '.', regex=False)
    data['price'] = pd.to_numeric(data['price'])
    data['active']=data['active'].str.strip().str.lower()=='yes'
    return data
     

In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame
# order_id, customer, city, payment без размака;
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    clean_df=df.copy()
    clean_df['order_id']=clean_df['order_id'].str.strip().str.upper()
    clean_df['customer']=clean_df['customer'].str.strip().str.title()
    clean_df['city']=clean_df['city'].str.strip().str.title()
    clean_df['payment']=clean_df['payment'].str.strip()
    clean_df['city'] = clean_df['city'].fillna('Unknown')
   
    return clean_df

In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return df[~is_dup], df[is_dup]

In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа
# 'OK' за исправан ред, у супротном разлог
# Провере радити редом и вратити први пронађени проблем
# order_id не почиње са NSS- -> 'invalid_order'
# quantity недостаје -> 'missing_quantity'
# quantity није цео број већи од нуле -> 'invalid_quantity'
# unit_price није број већи од нуле -> 'invalid_price'
# product_id не постоји међу производима -> 'unknown_product'
# Помоћ:
# pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
# pd.isna(value) препознаје и None и NaN
  
def validate_order(row: dict, product_ids: set) -> str:
    if row['order_id'].split('-')[0]!='NSS':
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    
    numeric_quantity=pd.to_numeric(row['quantity'], errors='coerce')

    if numeric_quantity<=0 or numeric_quantity!=int(numeric_quantity):
        return 'invalid_quantity'

     numeric_price=pd.to_numeric(row['unit_price'], errors='coerce')

    if pd.isna(numeric_price) or numeric_price<=0:
        return 'invalid_price'

    if row['product_id'] not in product_ids:
        return 'unknown_product'
    else: 
        return 'OK'


In [0]:
# 10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# pd.DataFrame
# Типови:
# quantity -> int
# unit_price -> float
# order_date -> датум (pd.to_datetime)
# Нова колона total (quantity * unit_price)
# Из очишћених производа (products) додати product_name, team_code и category
# (merge по product_id)
# Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df=df.copy()
    df['quantity']=df['quantity'].astype(int)
    df['unit_price']=df['unit_price'].astype(float)
    df['order_date']=pd.to_datetime(df['order_date'])
    df['total']=df['quantity']*df['unit_price']
    result = pd.merge(df, products, on='product_id').drop(columns=['sku','price','active'])
    return result



In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,
# rejected)
# valid су трансформисани исправни редови
# rejected су очишћени неисправни редови и дупликати, са колоном reason
# (дупликати за разлог треба да имају 'duplicate')
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df=clean_orders(df)
    unique,duplicates=remove_duplicate_orders(df)
    duplicates=duplicates.assign(reason='duplicate')
    product_ids=set(products['product_id'])
    reasons=unique.apply(lambda row: validate_order(row, product_ids), axis=1)
    valid=transform_orders(unique[reasons=='OK'], products)
    rejected=pd.concat([unique[reasons != 'OK'].assign(reason=reasons[reasons != 'OK']), duplicates])
    return valid, rejected


In [ ]:
# 12. Резултати тимова. Написати функцију team_results(matches: pd.DataFrame, teams:
# pd.DataFrame) -> pd.DataFrame која сваку одиграну утакмицу претвара у два реда, један
# за домаћина и један за госта.
# Користити само утакмице са status == 'finished' (одложена нема резултат)
# Колоне:
# match_id
# match_date (Датум из kickoff_ts, без времена - df['kickoff_ts'].dt.normalize())
# team_code, opponent_code
# goals_for, goals_against
# result ('W', 'D', 'L'), points (3, 1, 0)
# Утакмице имају team_id, а производи team_code
# codes = dict(zip(teams['team_id'], teams['code'])) # Прави речник облика
# {1: 'NPM', 2: 'FWR', ...}
# # map користи речник да претвори id у code
# home = pd.DataFrame(
# {
# ...
# 'team_code': played['home_team_id'].map(codes),
# 'opponent_code': played['away_team_id'].map(codes),
# ...
# })
#Output: 238 редова (119 одиграних утакмица * 2), са траженим колонама
# matches=podaci['matches']
# matches
# # teams=podaci['teams']
# # teams

def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    #prvo odvajam utakmice gde je status 'finished'
    played=matches[matches['status']=='finished'].copy()
    #pravim kljuc za sve timove iz df teams
    codes = dict(zip(teams['team_id'], teams['code']))
    #pravim kolonu sa rezultatima
    result_home=np.where(played['home_goals']>played['away_goals'], 'W',
                  np.where(played['home_goals']==played['away_goals'], 'D', 'L'))
    result_away=np.where(played['away_goals']>played['home_goals'], 'W',
                  np.where(played['home_goals']==played['away_goals'], 'D', 'L'))
    points_home=np.where(result_home == 'W', 3, np.where(result_home == 'D', 1, 0))
    points_away=np.where(result_away == 'W', 3, np.where(result_away == 'D', 1, 0))
    #df za home team
    home=pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].dt.date,
        'team_code': played['home_team_id'].map(codes),
        'opponent_code': played['away_team_id'].map(codes),
        'goals_for': played['home_goals'],
        'goals_against': played['away_goals'],
        'result': result_home,
        'points': points_home      
    }
    )
    #df za away team
    away=pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].dt.date,
        'team_code': played['away_team_id'].map(codes),
        'opponent_code': played['home_team_id'].map(codes),
        'goals_for': played['away_goals'],
        'goals_against': played['home_goals'],
        'result': result_away,
        'points': points_away      
    }
    )
    result=pd.concat([home, away], ignore_index=True)
    return result


In [ ]:
# 13. Табела лиге. Написати функцију league_table(results: pd.DataFrame) -> pd.DataFrame
# која од резултата из задатка 12 прави табелу
# Колоне:
# position, team_code
# played, won, drawn, lost
# goals_for, goals_against, goal_diff
# points
# Редослед табеле: Бодови, па гол разлика, па дати голови (све опадајуће)
# Output: 20 тимова, први FWR (26 бодова), други BMC (23)
# HGA и OKT имају по 11 утакмица

def league_table(results: pd.DataFrame) -> pd.DataFrame:
    #pravim najpre agregatne funkcije da izracunam sve to mi treba
    stats=results.groupby('team_code').agg(
        points=('points','sum'),
        goals_for=('goals_for', 'sum'),
        goals_against=('goals_against', 'sum'),
        played=('match_id', 'count'),
        won=('result', lambda x: (x == 'W').sum()),
        drawn=('result', lambda x: (x == 'D').sum()),
        lost=('result', lambda x: (x == 'L').sum())
    )

    stats['goal_diff']=stats['goals_for']-stats['goals_against']
    stats=stats.sort_values(['points', 'goal_diff', 'goals_for'], ascending=False)
    stats['position']=range(1, len(stats)+1)
    stats=stats.reset_index()
    stats=stats[[
        'position', 'team_code',
        'played', 'won', 'drawn', 'lost',
        'goals_for', 'goals_against', 'goal_diff',
        'points'
    ]]
    return stats

In [ ]:
# 14. Прозори после утакмица. Написати функцију match_windows(results: pd.DataFrame) ->
# pd.DataFrame која за сваки ред из задатка 12 прави 4 реда: дан утакмице и три дана после
# ње, са резултатом те утакмице
# Зашто: да бисмо поредили продају "у данима после победе" и "у данима после
# пораза", сваки дан мора да зна да ли је у прозору од 3 дана после утакмице свог тима
# и какав је био резултат.
# Пример: BRU је 2026-08-15 победио, па један ред из задатка 12 (team_results)
# постаје:
# team_code date window
# BRU 2026-08-15 W
# BRU 2026-08-16 W
# BRU 2026-08-17 W
# BRU 2026-08-18 W
# Помоћ:
# За сваки помак d од 0 до 3 (for d in range(4)) направити копију колона
# team_code, match_date, result и додати колону date померену за d дана
# Све 4 копије скупити у листу и спојити са pd.concat(..., ignore_index=True)
# Задржати колоне team_code, date, result и преименовати result у window
# Тимови играју најчешће једном у 6 дана, па се прозори једног тима никада не
# преклапају
# Output: 952 реда (238 * 4)
# колоне team_code, date, window
# четири реда за BRU од 2026-08-15 до 2026-08-18 су као у примеру изнад

def match_windows(results: pd.DataFrame) -> pd.DataFrame:

    copies=[]
    column_names=['team_code', 'match_date', 'result']

    for d in range(4):
        results_copy=results[column_names].copy()
        results_copy['date']=results_copy['match_date']+pd.Timedelta(days=d)
        copies.append(results_copy)
    window=pd.concat(copies, ignore_index=True)

    window=window[['team_code', 'date', 'result']]
    window=window.rename(columns={'result': 'window'})
    window=window.sort_values(['team_code', 'date']).reset_index(drop=True)
    return window

windows=match_windows(tr)
windows
#windows[windows['team_code']=='BRU']

In [ ]:
# 15. Промет по колони. Написати функцију revenue_by(df: pd.DataFrame, column: str) ->
# pd.DataFrame
# Kолоне [column, 'revenue'], сортирано опадајуће
# Input: valid, 'team_code' -> Output: 20 редова, први LSB 720420.00, последњи WFT
# 234650.00
# Input:valid, 'category' → Output: jerseys 7094410.00 на првом месту

def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:

    revenue=df.groupby(column)['total'].sum().reset_index()
    revenue=revenue.rename(columns={'total':'revenue'})
    revenue=revenue.sort_values('revenue', ascending=False)
    return revenue

# r=revenue_by(valid, 'team_code')
# r
r=revenue_by(valid, 'category')
r

In [ ]:
# 16. Продаја после резултата. Написати функцију sales_after_results(valid: pd.DataFrame,
# windows: pd.DataFrame) -> pd.DataFrame која одговара на питање пројекта
# 1. Календар: Сваки тим x Сваки дан.
# # 20 teams
# teams = pd.DataFrame({'team_code': windows['team_code'].unique()})
# # 93 days (every day has orders)
# days = pd.DataFrame({'order_date': valid['order_date'].unique()})
# # 20 x 93 = 1,860 rows
# calendar = teams.merge(days, how='cross')
# 2. Дневни промет по тиму и дану
# valid.groupby(['team_code', 'order_date'])
# ['total'].sum().reset_index(name='revenue')
# Спојити га на календар (дан без продаје добија NaN за revenue, замените то са
# 0)
# 3. Спојити календар са прозорима добијених из match_windows
# calendar.merge(windows, left_on=['team_code', 'order_date'],
# right_on=['team_code', 'date'], how='left')
# 4. Дан који није ни у једном прозору нема резултат и добија 'none'
# calendar['window'] = calendar['window'].fillna('none')
# 5. Груписати по window: team_days (број дана), revenue (збир), avg_daily_revenue
# (просек, 2 децимале), сортирано опадајуће
# Output:
# window team_days revenue avg_daily_revenue
# W 339 2999230.0 8847.29
# D 228 1180180.0 5176.23
# none 954 4427150.0 4640.62
# L 339 1012330.0 2986.22

def sales_after_results(valid: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:

    # 1. Календар: Сваки тим x Сваки дан.
    #ovde su mi jedinstveni team_codes iz 14. zadatka (windows - za svaki tim, dan utakmice i tri posle)
    teams = pd.DataFrame({'team_code': windows['team_code'].unique()})
    #93 days (every day has orders) jedinstveni datumi porudzbina
    days = pd.DataFrame({'order_date': valid['order_date'].unique()})
    #20 x 93 = 1,860 rows, Dekartov proizvod redova ova dva datafrejma
    calendar = teams.merge(days, how='cross') #imam red za svaki tim, za svaki datum
    #return calendar

    #2. Дневни промет по тиму и дану
    daily_revenue=valid.groupby(['team_code', 'order_date'])['total'].sum().reset_index(name='revenue')
    calendar=calendar.merge(daily_revenue, 
                            on=['team_code', 'order_date'],
                            how='left')  #svaki red je dobio svoj revenue, dakle po timu po danu
    calendar['revenue']=calendar['revenue'].fillna(0)
    # return calendar[calendar['revenue']==0] 757 redova bez prodaje

    # 3. Спојити календар са прозорима добијених из match_windows
    calendar['order_date']=pd.to_datetime(calendar['order_date'])
    windows['date']=pd.to_datetime(windows['date'])
    calendar=calendar.merge(windows, 
                    left_on=['team_code', 'order_date'],
                    right_on=['team_code', 'date'], 
                    how='left')
    # return calendar

    # 4. Дан који није ни у једном прозору нема резултат и добија 'none'

    calendar['window'] = calendar['window'].fillna('none')
    # return calendar

    # 5. Груписати по window: team_days (број дана), revenue (збир), avg_daily_revenue
    # (просек, 2 децимале), сортирано опадајуће

    result=calendar.groupby('window').agg(
        team_days=('team_code', 'count'),
        revenue=('revenue', 'sum'),
        avg_daily_revenue=('revenue', 'mean')
    ).reset_index()
    result=round(result, 2)
    result=result.sort_values('revenue', ascending=False)
    return result
    
sales=sales_after_results(valid, windows)
sales

In [0]:
# 17. Load. Написати функцију load(outputs: dict, processed_dir: str) -> dict која сваки
# DataFrame из речника {име_фајла: DataFrame} уписује у processed_dir и враћа
# {име_фајла: број_редова}

def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
    if name.endswith('.parquet'):
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
        written[name] = len(df)
    return written

In [0]:
# 18. Написати функцију run_pipeline(raw_dir: str, processed_dir: str) -> dict која
# покреће цео pipeline

def run_pipeline(raw_dir: str, processed_dir: str) -> None:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=
        ['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric',
    'value']).to_csv(
    os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary

run_pipeline(RAW_DIR, PROCESSED_DIR)


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
~/.ipykernel/74/command-7320242248470469-1951368561 in ?()
     30     'value']).to_csv(
     31     os.path.join(processed_dir, 'summary.csv'), index=False)
     32     return summary
     33 
---> 34 run_pipeline(RAW_DIR, PROCESSED_DIR)

~/.ipykernel/74/command-7320242248470469-1951368561 in ?(raw_dir, processed_dir)
      4 def run_pipeline(raw_dir: str, processed_dir: str) -> None:
      5     data = extract(raw_dir)
----> 6     products = clean_products(data['products'])
      7     valid, rejected = split_orders(data['orders'], products)
      8     results = team_results(data['matches'], data['teams'])
      9     windows = match_windows(results)

~/.ipykernel/74/command-7320242248470477-3580543297 in ?(df)
     16 def clean_products(df: pd.DataFrame) -> pd.DataFrame:
     17     df=df.copy()
     18     data=remove_duplicate_pro